# 08 — Visual Analytics Interactivo

**TFM — Valoración Inmobiliaria Masiva: SLX vs Machine Learning**

---

Genera los dos artefactos HTML interactivos para la defensa del TFM.

### Entregables

| Artefacto | Ruta |
|-----------|------|
| Mapa comparativo de residuos | `outputs/visuals/mapa_comparativo_residuos.html` |
| SHAP force plot · 200 viviendas | `outputs/visuals/shap_explicabilidad.html` |
| SHAP force plot · vivienda cara | `outputs/visuals/shap_vivienda_cara.html` |
| SHAP force plot · vivienda media | `outputs/visuals/shap_vivienda_media.html` |
| SHAP force plot · vivienda barata | `outputs/visuals/shap_vivienda_barata.html` |

### Contenido
0. Setup e imports
1. Carga y preparación de datos
2. **Mapa comparativo de residuos** — Folium, 3 capas intercambiables
3. **Interpretabilidad SHAP interactiva** — force plot individual y múltiple
4. Resumen de artefactos

---
## 0. Setup e imports

In [ ]:
import sys
import os
import time
import warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import joblib
import shap
import folium
import branca.colormap as cm
from pathlib import Path

PROCESSED = '../data/processed/'
VISUALS   = '../outputs/visuals/'
os.makedirs(VISUALS, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print('Entorno listo.')
print(f'  folium : {folium.__version__}')
print(f'  shap   : {shap.__version__}')

---
## 1. Carga y preparación de datos

Unificamos las predicciones de los tres modelos con las coordenadas del test.  
Los precios se convierten de log-scale a dólares con `np.expm1()`.

In [ ]:
coords   = pd.read_csv(PROCESSED + 'coords_test.csv')
pred_rf  = pd.read_csv(PROCESSED + 'pred_rf.csv')
pred_xgb = pd.read_csv(PROCESSED + 'pred_xgb.csv')
pred_slx = pd.read_csv(PROCESSED + 'pred_slx.csv')
X_test   = pd.read_csv(PROCESSED + 'X_test_spatial.csv')

df = coords.copy()
df['y_real']         = pred_rf['y_real']
df['price_real']     = np.expm1(df['y_real'])
df['price_pred_slx'] = np.expm1(pred_slx['y_pred_slx'])
df['price_pred_rf']  = np.expm1(pred_rf['y_pred_rf'])
df['price_pred_xgb'] = np.expm1(pred_xgb['y_pred_xgb'])
df['error_slx']      = (df['price_real'] - df['price_pred_slx']).abs()
df['error_rf']       = (df['price_real'] - df['price_pred_rf']).abs()
df['error_xgb']      = (df['price_real'] - df['price_pred_xgb']).abs()

print(f'Viviendas en test: {len(df):,}')
print(f'\n{"Modelo":<22} {"Error medio":>14} {"Mediana":>14} {"p95":>14}')
print('-' * 66)
for nombre, col in [('SLX (Econométrico)', 'error_slx'),
                     ('Random Forest',      'error_rf'),
                     ('XGBoost',            'error_xgb')]:
    print(f'{nombre:<22} ${df[col].mean():>12,.0f} ${df[col].median():>12,.0f} ${df[col].quantile(0.95):>12,.0f}')

---
## 2. Mapa Comparativo de Residuos

Mapa interactivo Folium con **3 capas** (SLX · Random Forest · XGBoost).  
Cada vivienda es un círculo coloreado según su **error absoluto en dólares**:

- **Amarillo / naranja claro** → error bajo (modelo preciso)
- **Naranja intenso / rojo** → error alto (modelo falla)

El `LayerControl` (esquina superior derecha) permite cambiar de modelo.  
Ver cómo las manchas rojas del SLX desaparecen al cambiar a XGBoost es el argumento visual clave del TFM.

> La escala de color usa el percentil 95 del error XGBoost como máximo, lo que hace que XGBoost aparezca predominantemente en amarillo/naranja suave y SLX en rojo intenso.

> ⚠️ La generación puede tardar **3–8 minutos** por los ~13 000 marcadores.

In [ ]:
p95 = df['error_xgb'].quantile(0.95)
print(f'Escala de color: $0 — ${p95:,.0f}  (percentil 95 XGBoost)')

colormap = cm.LinearColormap(
    colors=['#ffffb2', '#fecc5c', '#fd8d3c', '#f03b20', '#bd0026'],
    vmin=0, vmax=p95,
    caption='Error Absoluto (USD)'
)

# ── Mapa base ──────────────────────────────────────────────────────────────
centro = [float(df['lat'].mean()), float(df['long'].mean())]
mapa = folium.Map(location=centro, zoom_start=10, tiles='CartoDB positron')
colormap.add_to(mapa)

title_html = (
    "<div style='position:fixed;top:12px;left:50%;transform:translateX(-50%);"
    "z-index:9999;background:rgba(255,255,255,0.95);padding:10px 20px;"
    "border-radius:10px;box-shadow:0 2px 12px rgba(0,0,0,0.25);"
    "font-family:Arial,sans-serif;text-align:center;white-space:nowrap'>"
    "<b style='font-size:14px;color:#1a202c'>Comparativa de Residuos &mdash; SLX vs RF vs XGBoost</b><br>"
    "<span style='font-size:11px;color:#718096'>King County, WA &nbsp;&middot;&nbsp; "
    "usa el control de capas para cambiar de modelo</span>"
    "</div>"
)
mapa.get_root().html.add_child(folium.Element(title_html))

# ── Capas por modelo ────────────────────────────────────────────────────────
capas = [
    ('SLX (Econometrico)',  'error_slx', 'price_pred_slx', True),
    ('Random Forest',      'error_rf',  'price_pred_rf',  False),
    ('XGBoost',            'error_xgb', 'price_pred_xgb', False),
]

for model_name, error_col, pred_col, show in capas:
    t0 = time.time()
    print(f'Generando capa: {model_name}...', end='', flush=True)
    group = folium.FeatureGroup(name=model_name, show=show)

    for _, row in df.iterrows():
        err     = float(min(row[error_col], colormap.vmax))
        color   = colormap(err)
        opacity = 0.15 + 0.85 * (err / colormap.vmax)

        tooltip_html = (
            f"<div style='font-family:Arial;font-size:12px;line-height:1.7'>"
            f"<b style='color:#2d3748;font-size:13px'>{model_name}</b>"
            f"<hr style='margin:4px 0;border-color:#e2e8f0'>"
            f"Real:     <b>${row['price_real']:>10,.0f}</b><br>"
            f"Predicho: ${row[pred_col]:>10,.0f}<br>"
            f"Error: <b style='color:#bd0026'>${row[error_col]:>10,.0f}</b>"
            f"</div>"
        )

        folium.CircleMarker(
            location=[float(row['lat']), float(row['long'])],
            radius=5,
            weight=0,
            fill=True,
            fill_color=color,
            fill_opacity=float(opacity),
            tooltip=folium.Tooltip(tooltip_html, sticky=True)
        ).add_to(group)

    group.add_to(mapa)
    print(f'  ({time.time()-t0:.0f}s)')

folium.LayerControl(collapsed=False, position='topright').add_to(mapa)

output_map = VISUALS + 'mapa_comparativo_residuos.html'
mapa.save(output_map)

# ── Post-proceso: convertir checkboxes en radio buttons ─────────────────────
# Folium genera checkboxes independientes por defecto (permite seleccionar
# varias capas a la vez). Este snippet los convierte en radio buttons del
# mismo grupo — Leaflet sigue escuchando los eventos 'change' y funciona igual.
radio_script = """
<script>
setTimeout(function () {
  var overlays = document.querySelector('.leaflet-control-layers-overlays');
  if (!overlays) return;
  overlays.querySelectorAll('input[type="checkbox"]').forEach(function (cb) {
    cb.type = 'radio';
    cb.name = 'modelo';
  });
}, 600);
</script>
"""
html_text = Path(output_map).read_text(encoding='utf-8')
html_text = html_text.replace('</body>', radio_script + '\n</body>')
Path(output_map).write_text(html_text, encoding='utf-8')

size_mb = Path(output_map).stat().st_size / 1024 / 1024
print(f'\nMapa guardado: {output_map}  ({size_mb:.1f} MB)')
print('Solo un modelo activo a la vez (radio buttons). Abre en Chrome/Firefox.')

---
## 3. Interpretabilidad SHAP Interactiva

El **TreeExplainer** descompone exactamente cada predicción del modelo XGBoost:

$$\hat{y}_i = \underbrace{\phi_0}_{\text{valor base}} + \sum_{j=1}^{30} \underbrace{\phi_{ij}}_{\text{contribución SHAP}}$$

- **Rojo** → feature empuja el precio hacia **arriba**
- **Azul** → feature empuja el precio hacia **abajo**

Usamos **200 observaciones** del test (tiempo razonable, representatividad estadística garantizada).

In [ ]:
xgb_model = joblib.load('../outputs/models/xgb_final.joblib')

np.random.seed(SEED)
sample_idx = np.random.choice(len(X_test), 200, replace=False)
X_sample   = X_test.iloc[sample_idx].reset_index(drop=True)
y_pred_sample = pred_xgb['y_pred_xgb'].iloc[sample_idx].reset_index(drop=True)

print('Calculando SHAP values sobre 200 observaciones...')
t0 = time.time()
explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_sample)
expected_v = explainer.expected_value
print(f'  Completado en {time.time()-t0:.1f}s')
print(f'\nShap values shape : {shap_vals.shape}')
print(f'Expected value    : {expected_v:.4f}  (log_price medio del train)')
print(f'Rango contribuciones: [{shap_vals.min():.4f}, {shap_vals.max():.4f}]')

pred_check = xgb_model.predict(X_sample)
max_err    = np.abs(pred_check - (expected_v + shap_vals.sum(axis=1))).max()
print(f'\nVerificacion: max |pred - (base + SHAP)| = {max_err:.6f}  (esperado ~0)')

shap.initjs()

### 3.1 Force Plot individual — tres viviendas representativas

Seleccionamos la vivienda **más cara**, la **más barata** y la de **precio mediano** de la muestra.  
El force plot muestra cómo cada feature contribuye a desplazar el precio desde el valor base $\phi_0$.

In [ ]:
pred_usd = np.expm1(y_pred_sample.values)
idx_cara   = int(np.argmax(pred_usd))
idx_barata = int(np.argmin(pred_usd))
idx_media  = int(np.argmin(np.abs(pred_usd - np.median(pred_usd))))

feat_names = X_sample.columns.tolist()

casos = [
    ('cara',   idx_cara,   'Vivienda mas cara'),
    ('media',  idx_media,  'Vivienda con precio mediano'),
    ('barata', idx_barata, 'Vivienda mas barata'),
]

for nombre, pos, titulo in casos:
    precio_usd = int(np.expm1(y_pred_sample.iloc[pos]))
    fp = shap.force_plot(
        expected_v,
        shap_vals[pos],
        X_sample.iloc[pos],
        feature_names=feat_names,
        matplotlib=False
    )
    ruta = VISUALS + f'shap_vivienda_{nombre}.html'
    shap.save_html(ruta, fp)
    print(f'{titulo}: ${precio_usd:,}  ->  {ruta}')

print('\nVisualizando la vivienda mediana en el notebook:')
shap.force_plot(expected_v, shap_vals[idx_media], X_sample.iloc[idx_media],
                feature_names=feat_names)

### 3.2 Force Plot múltiple — 200 viviendas (artefacto principal)

Este es el gráfico **más impactante para la defensa**. Al pasar las 200 observaciones al force plot, SHAP genera una visualización apilada e interactiva:

- Cada fila horizontal es una vivienda
- El color muestra qué features elevan (rojo) o reducen (azul) el precio predicho
- El **deslizador interactivo** permite **ordenar las 200 viviendas por cualquier feature**  
  (prueba a ordenar por `grade` o `lag_grade_k15` para ver la segregación del mercado)

> Abre `outputs/visuals/shap_explicabilidad.html` en Chrome y usa el desplegable para cambiar el eje de ordenación.

In [ ]:
print('Generando force plot para 200 viviendas...')\nt0 = time.time()\n\nforce_multi = shap.force_plot(\n    expected_v,\n    shap_vals,\n    X_sample,\n    feature_names=feat_names,\n    matplotlib=False\n)\n\noutput_shap = VISUALS + 'shap_explicabilidad.html'\nshap.save_html(output_shap, force_multi)\nprint(f'Completado en {time.time()-t0:.1f}s')\n\n# ── Post-proceso: fix overflow + padding derecho ────────────────────────────\n# El force plot de SHAP recorta los números del extremo derecho cuando la\n# ventana no es suficientemente ancha. Este CSS lo soluciona.\n_fix_css = \"\"\"<style>\nhtml, body { overflow-x: auto; }\nsvg { overflow: visible !important; }\nbody > div { padding-right: 120px !important; box-sizing: border-box; }\n</style>\"\"\"\n\n# Aplicar a todos los HTML SHAP generados\nfor _ruta in [\n    VISUALS + 'shap_explicabilidad.html',\n    VISUALS + 'shap_vivienda_cara.html',\n    VISUALS + 'shap_vivienda_media.html',\n    VISUALS + 'shap_vivienda_barata.html',\n]:\n    _p = Path(_ruta)\n    if not _p.exists():\n        continue\n    _html = _p.read_text(encoding='utf-8')\n    if 'overflow-x: auto' not in _html:\n        _html = _html.replace('<head>', '<head>' + _fix_css, 1)\n        _p.write_text(_html, encoding='utf-8')\n\nsize_kb = Path(output_shap).stat().st_size / 1024\nprint(f'\\nHTML guardado: {output_shap}  ({size_kb:.0f} KB)')\nprint('Abre en Chrome y usa el desplegable para ordenar por feature.')\n\nforce_multi

---
## 4. Resumen de artefactos generados

| Artefacto | Descripcion |
|-----------|-------------|
| `mapa_comparativo_residuos.html` | Mapa Folium: 3 capas (SLX/RF/XGBoost), circulo = vivienda, color = error absoluto |
| `shap_explicabilidad.html` | Force plot 200 viviendas: interactivo, ordena por cualquier feature |
| `shap_vivienda_cara.html` | Force plot individual: vivienda mas cara de la muestra |
| `shap_vivienda_media.html` | Force plot individual: vivienda con precio mediano |
| `shap_vivienda_barata.html` | Force plot individual: vivienda mas barata de la muestra |

In [ ]:
from pathlib import Path as _P

artefactos = sorted(_P(VISUALS).glob('*.html'))

print('=== Artefactos en outputs/visuals/ ===')
for f in artefactos:
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<45s}  {size_kb:>7.0f} KB')
print(f'\nTotal: {len(artefactos)} archivos HTML')